# ARCSIX single-granule all-liquid reference test

Target from the supplied figure: **June 10, 2024, 15:42–15:47 UTC**. NASA CMR confirms HARP2 GPC V4.0 and OCI L1C V3 at **20240610T154205**, covering 15:42:05–15:47:04 UTC. The flight track is not used.

Run cells in order in AWS. This notebook prepares exactly one HARP2/OCI L1C pair, then its native OCI cloud retrievals and original ancillary inputs. Adjacent native OCI L2 granules may be needed for observation-time matching; that does not add other L1C pairs.

The 265 K, 3 m/s ocean LUT is applied globally, including land/ice. Both OCI CER/COT bands and HARP2 CER alternatives are retained. Near-surface profile exclusions and the other limitations in `../../Documentation/algorithm_details.md` still apply. This is a setup for your run, not an already executed Arctic retrieval.

The figure's separate “mixed-phase” category is not implemented: this test keeps our current three classes (liquid, ice, LTMP).

In [ ]:
# Run only if these dependencies are missing in this environment:
# Install from the repository root in this kernel environment:
# python -m pip install -e ".[validation]"

from pathlib import Path
from datetime import date, timedelta
from dataclasses import replace
import json
import shutil
import numpy as np
import xarray as xr
import earthaccess
from pace_specpol import matching as mp
from pace_specpol.reference import Reference
from pace_specpol.validation.aggregation import IndexConfig, PhaseIndex, build_index
from pace_specpol.validation.vis_dashboard import dashboard, reference_dashboard
from pace_specpol.validation.vis_samples import plot_cached_samples
from pace_specpol.paths import load_paths
PATHS = load_paths()
from pace_specpol.liquid_reference import LiquidConfig
from pace_specpol.oci_companion import CompanionConfig, OCICompanion, discover_cloud, atomic_json
from pace_specpol.workflow import (
    cache_companions, cache_references, default_variants,
    VariantReference,
)



## Local input check, then single-granule extraction

Run the local configuration and LUT-file check below before Earthdata login or satellite discovery. It checks file presence without loading the large tables. If an input is missing, obtain the tables separately and update `lut_root` in your local configuration; no automatic LUT download is provided.

Only the exact catalog timestamp below is selected. Separate caches keep this test independent of the September run. The full selected granule is retained; the Arctic extent changes only map display, not the histogram population.


In [ ]:
LUT_ROOT = PATHS["lut_root"]
WORK_ROOT = PATHS["work_root"]
MAX_PAIRS = 1             # None = all cached pairs
EXPERIMENT = "arcsix_20240610T154205_global_ocean3_265K_v1"  # change for new LUT or scientific settings
RUN = f"{EXPERIMENT}_pilot{MAX_PAIRS}" if MAX_PAIRS is not None else f"{EXPERIMENT}_full"
COMPANION_DIR = WORK_ROOT / RUN / "companions"
REFERENCE_DIR = WORK_ROOT / RUN / "references"

liquid_config = LiquidConfig(
    ms_path=str(LUT_ROOT / "LIQUID/ocean_msr_water_wspeed_3_v6.PACE.1.1.5.2026144071240.hdf"),
    phase_path=str(LUT_ROOT / "IceAndWaterPhaseFunctionData_v6.PACE.1.1.5.2026142144440.hdf"),
    transmittance_path=str(LUT_ROOT / "Transmittance_OCI.hdf"),
    cer_source="oci",
    oci_reference_band=2260,   # change to 2130 when selecting one mode
    oci_phase_policy="all",   # "liquid" = OCI phase code 2 only
    ocean_only=False,
)
try:
    liquid_config.validate()
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"Required LUT input is missing: {exc}. "
        "Obtain the supplied 265 K LUT set and transmission table, then "
        "update lut_root in your local TOML configuration. "
        "See Documentation/data_and_luts.md. No satellite downloads have started."
    ) from exc
print("Required LUT files found. Ready for satellite discovery/extraction.")


In [ ]:
TARGET_STAMP = "20240610T154205"
PAIR_CACHE = PATHS["pair_cache"]
earthaccess.login()
match_config = mp.MatchConfig(
    start_date="2024-06-10", end_date="2024-06-10",
    cache_dir=str(PAIR_CACHE), workers=1, batch_size=1,
    allow_unpaired=True,  # unrelated gaps elsewhere that day do not block this test
)
all_pairs, catalogs, discovery_report = mp.discover_pairs(match_config)
pairs = [p for p in all_pairs if p["stamp"] == TARGET_STAMP]
if len(pairs) != 1:
    raise RuntimeError(f"Expected exactly one refined pair at {TARGET_STAMP}; found {len(pairs)}")
print("Selected HARP2:", pairs[0]["harp"]["name"])
print("Selected OCI:", pairs[0]["oci"]["name"])
discovery_report.update(selected_stamp=TARGET_STAMP, selected_pairs=1,
                        selection="exact ARCSIX figure granule; full granule retained")
mp.cache_pairs(pairs, catalogs, match_config, discovery_report)


In [ ]:
manifest = json.loads((PAIR_CACHE / "manifest.json").read_text())
assert manifest["complete"], "Finish the original paired cache first."
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("Paired files:", len(manifest["files"]))
print("Free disk GiB:", round(shutil.disk_usage(WORK_ROOT).free / 2**30, 2))
print("Raw paired data are read-only inputs. Output:", REFERENCE_DIR)


## Discover refined OCI cloud retrievals

The new reader corrects legacy L1C observation timestamps using NASA's offset sign and searches adjacent CLD granules. Discovery includes a day of padding on either side; accepted samples still obey the original requested dates.

For AWS, `transport="earthaccess"` uses S3. `transport="https"` uses public cloud range reads and is also supported. The much smaller original meteorological files are fetched separately through OB.DAAC, using the exact names recorded in each CLD product. No replacement GEOS product is silently selected.

In [ ]:
earthaccess.login()
start = date.fromisoformat(manifest["config"]["start_date"])
end = date.fromisoformat(manifest["config"]["end_date"])
cloud_version = "3.1"
catalog_path = WORK_ROOT / f"OCI_CLD_{start}_{end}_V{cloud_version}.json"
if catalog_path.exists():
    catalog = json.loads(catalog_path.read_text())
else:
    catalog = discover_cloud(str(start-timedelta(days=1)), str(end+timedelta(days=1)), cloud_version)
    atomic_json(catalog_path, catalog)
print("Refined CLD granules, including padding:", len(catalog))

companion_config = CompanionConfig(
    cloud_version=cloud_version,
    transport="earthaccess",  # "https" is an alternative
    ancillary_cache=str(WORK_ROOT / "ancillary_downloads"),
    max_download_cache_mb=512,
    minimum_free_disk_mb=512,
    max_distance_km=3.0,
    max_time_difference_s=10.0,
)
provider = OCICompanion(
    catalog, companion_config,
    session=earthaccess.get_requests_https_session(),
)


## Stage 1: cache OCI microphysics and above-cloud water

This is the remote-data stage. It processes one paired granule at a time and writes after each. If a network or authentication error occurs, log in again, recreate `provider`, and rerun this cell. Completed companions are reused.

The original cache's erroneous date-boundary exclusions cannot be undone from cached samples alone; existing in-range samples are corrected and retained. See the README before using a legacy cache for precise boundary accounting.

In [ ]:
cache_companions(PAIR_CACHE, COMPANION_DIR, provider, max_pairs=MAX_PAIRS)


## Stage 2: all-liquid references (local computation)

Both OCI modes use their matching COT/CER pair. HARP2 modes replace CER only. The same atmospheric correction is used throughout. An ice-derived OCI CER is treated explicitly as a same-numerical-radius liquid counterfactual; out-of-range CER is rejected, not clamped.

The dictionary below controls which variants are computed. The common population is the intersection valid in every listed variant. For a two-mode comparison, retain only those two dictionary entries before creating a new reference directory.

In [ ]:
variants = default_variants(liquid_config)
# Example: only compare CER sources with OCI 2260 COT:
# variants = {k: variants[k] for k in ("oci_2260", "harp2_2260")}

cache_references(COMPANION_DIR, REFERENCE_DIR, variants)


## Inspect exclusions before interpreting phase maps

Reason counts overlap: one sample can fail more than one check. Excluded pixels are missing, never relabeled as ice or liquid. Differences in each mode's valid population must not be interpreted as phase changes.

In [ ]:
report = json.loads((REFERENCE_DIR / "manifest.json").read_text())
for variant in variants:
    records = [r["variants"][variant] for r in report["records"]]
    print(variant, "valid:", sum(r["valid"] for r in records),
          "ice CER used as liquid:", sum(r["ice_cer_as_liquid_valid"] for r in records))
    print({reason: sum(r["reason_counts"][reason] for r in records)
           for reason in records[0]["reason_counts"]})
print("Common valid:", sum(r["common_valid"] for r in report["records"]))
print("Free disk GiB:", round(shutil.disk_usage(WORK_ROOT).free / 2**30, 2))


## Interactive comparison

Choose CER/COT mode, sample population, and ratio space; click **Show selection**. The first selection builds an on-disk sparse index. Later use reopens it. Threshold changes inside the dashboard need no remote reads or LUT recomputation.

1.27 remains a provisional threshold, not a newly calibrated OCI mixed-phase boundary. Compare the common population to isolate reference changes. The maps show modal phase plus mean LI and mean ratio; rare phases can be hidden by the modal map.

The reversible `layout="slide"` preset exports a 16:9 figure (13⅓ × 7.5 inches). `map_framing="granule"` fits occupied cells vertically with padding and retains/expands horizontal context; it does not crop the histogram or exported data. Set `layout="classic", map_framing="extent"` to restore the prior layout/framing. See `DASHBOARD_LAYOUT.md`.

In [ ]:
reference_dashboard(
    REFERENCE_DIR,
    index_config=IndexConfig(resolution=0.1, ratio_min=0.5, ratio_max=3.5, ratio_step=0.01),
    threshold=1.27,
    layout="slide",          # "classic" restores the original 16 x 12 layout
    map_framing="granule",   # "extent" restores the original geographic framing
    graticules=True,         # False removes geographic lines/labels
    map_extent=(-85, -10, 58, 87),  # regional polar display; full-granule histogram
    ratio_clim=(0.5, 2.0),
    li_clim=(-0.5, 4.0),
    output_dir=str(PATHS["export_root"] / RUN),
)


## Optional full-resolution export for a selected experiment

Exports retain class counts and fractions as well as dominant phase. Use a descriptive filename. This cell reads the prepared reference cache only.

In [ ]:
SELECTED_VARIANT = "oci_2260"  # "oci_2130", "harp2_2260", "harp2_2130"
SELECTED_POPULATION = "own"    # "common"
SELECTED_THRESHOLD = 1.27
# Uncomment to build/export:
# index = build_index(REFERENCE_DIR,
#     reference=VariantReference(SELECTED_VARIANT, SELECTED_POPULATION),
#     config=IndexConfig(resolution=0.1))
# output = WORK_ROOT / RUN / f"{SELECTED_VARIANT}_{SELECTED_POPULATION}_phase.nc"
# index.export(output, threshold=SELECTED_THRESHOLD)


# ARCSIX airborne validation intercomparison
The following cells use the reference-cache stage above. They require its `REFERENCE_DIR`, `WORK_ROOT`, and `RUN` variables and the installed `pace_specpol` package.

**Verified June 10 product:** `ARCSIX-HSRL-CloudAndSurface_G3_20240610_R1_L1.h5`, in the HALO collection (approximately 324 MiB). Actual navigation spans **10:58:06–15:24:12.572 UTC**. This ends about 18 minutes before the satellite granule; close time-window matches may correctly be empty. No advection or cloud parallax correction is performed. Distances use aircraft GPS and satellite centers, not exact footprint polygons.

The file's phase codes are 0=water-dominant, 1=ice/water OR ice/horizontally oriented ice (HOI), 2=ice-dominant, 3=HOI, 4=aerosol. Code 1 is **not confirmed mixed phase**. The top-window summary below is our configurable diagnostic, not a new official NASA phase product. Liquid tops with unobserved cloud interiors remain unknown for LTMP truth. The archived readme encourages consultation with the lidar team; no uncertainty fields are supplied in this release.

[HALO collection](https://asdc.larc.nasa.gov/project/ARCSIX/ARCSIX_AircraftRemoteSensing_LaRC-G3_HALO_Data_1) · [ARCSIX archive](https://www-air.larc.nasa.gov/missions/arcsix/)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pace_specpol.validation import arcsix as av

VALIDATION_DIR = WORK_ROOT / RUN / "airborne_validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
AIRBORNE_DIR = PATHS["airborne_root"]
DAY = "2024-06-10"
VARIANT = "oci_2260"  # also oci_2130, harp2_2260, harp2_2130
POPULATION = "common"  # same population for comparisons across variants
MAX_DISTANCE_KM = 3.0  # center-distance proxy; test 1, 3, 5 km
TIME_WINDOWS_MIN = [5, 10, 20, 30, 60]
SELECTED_WINDOW_MIN = 20  # exploratory; never silently broadened
TOP_DEPTH_M = 90
LI_CUTOFF = 0.3
RATIO_CUTOFF = 1.27

earthaccess.login()
air_catalog = av.discover(DAY)
for kind, entries in air_catalog.items():
    print(kind, len(entries))
    for g in entries:
        if kind in ("lidar", "navigation"): print(" ", av.name(g))
# Save catalog metadata, not temporary signed download URLs.
(VALIDATION_DIR / "airborne_catalog.json").write_text(json.dumps(air_catalog, default=dict, indent=2))

from pace_specpol.validation.vis_airborne import plot_matchup_context, plot_lidar_curtain


## Load the cloud-phase profiles and satellite samples
Only the cloud/surface HDF5 file is downloaded here—not the large particle-image archives. Reading the phase mask uses small chunks. Each top-window label requires at least three classified cloud bins and 75% agreement, with no ambiguous/HOI bins in that window. Change the depth and purity to test label sensitivity. The “ice below” diagnostic is evidence only: it can include another layer and does not establish absence of ice when zero.

The phase window starts at the **highest classified cloud-mask bin below the aircraft and above the surface**. The separate “Cloud Top Height” property-retrieval field can refer to a lower water layer beneath ice, so it is retained as `retrieval_cloud_top_m` and is not used as the uppermost phase boundary.

In [ ]:
hsrl_path = av.download_exact(
    air_catalog["lidar"], "ARCSIX-HSRL-CloudAndSurface_G3_20240610_R1_L1.h5", AIRBORNE_DIR)
air = av.read_hsrl(hsrl_path, DAY, top_depth_m=TOP_DEPTH_M, min_bins=3, purity=0.75)
sat = av.read_satellite(REFERENCE_DIR, VARIANT, POPULATION)
print("Aircraft record:", air.time.min(), "to", air.time.max())
print("OCI sample times:", sat.time.min(), "to", sat.time.max())
print("HARP nadir times:", sat.harp_time.min(), "to", sat.harp_time.max())
print("Aircraft end to earliest OCI sample (minutes):", (sat.time.min()-air.time.max()).total_seconds()/60)
display(air.lidar_top.value_counts())


## Quantify overlap before plotting or tuning
Matching uses OCI observation times. Each aircraft profile is assigned to the closest spatially eligible satellite sample within the chosen time window; no ratio or phase value participates in matching. Geometry matches are retained even when the LUT reference is invalid. Many airborne profiles in one satellite sample are collapsed to one comparison record, with all label fractions retained. This avoids counting hundreds of profiles as hundreds of independent satellite pixels, although adjacent satellite samples are still correlated.

In [ ]:
overlap = []
selected_matches = None
for window in TIME_WINDOWS_MIN:
    matches = av.match_profiles(sat, air, max_minutes=window, max_km=MAX_DISTANCE_KM)
    summary = av.aggregate_matches(matches, min_profiles=3, purity=0.8)
    overlap.append(dict(window_min=window, profiles=len(matches), satellite_samples=len(summary),
        confident_top_samples=0 if summary.empty else int(summary.lidar_top.isin(["water_dominant","ice_dominant"]).sum())))
    if window == SELECTED_WINDOW_MIN: selected_matches = matches
if selected_matches is None:
    selected_matches = av.match_profiles(sat, air, SELECTED_WINDOW_MIN, MAX_DISTANCE_KM)
footprints = av.aggregate_matches(selected_matches, min_profiles=3, purity=0.8)
display(pd.DataFrame(overlap))
if footprints.empty:
    print("No matchups at the selected tolerances. Do not infer phase skill from this case.")
else:
    display(footprints[["sat_id","n_profiles","dt_median_min","distance_max_km","lidar_top","valid"]].head())
tag = f"{VARIANT}_{POPULATION}_{SELECTED_WINDOW_MIN}min_{MAX_DISTANCE_KM:g}km"
selected_matches.to_csv(VALIDATION_DIR / f"{tag}_profile_matches.csv", index=False)
footprints.to_csv(VALIDATION_DIR / f"{tag}_satellite_comparisons.csv", index=False)
(VALIDATION_DIR / f"{tag}_settings.json").write_text(json.dumps(dict(day=DAY,variant=VARIANT,
    population=POPULATION,max_distance_km=MAX_DISTANCE_KM,time_window_min=SELECTED_WINDOW_MIN,
    top_depth_m=TOP_DEPTH_M,hsrl_file=hsrl_path.name,reference_dir=str(REFERENCE_DIR),
    geometry="center distance; aircraft GPS; no advection or cloud-parallax correction"),indent=2))


## Track map and matched phase distributions
The map distinguishes the full lidar flight, the selected satellite granule, and accepted matches. The joint plot colors observations by lidar **cloud-top** evidence; it is not an LTMP truth plot. The two lower traces use satellite sample time; aircraft time offsets are in the exported table.

In [ ]:
plot_matchup_context(sat, air, selected_matches, footprints, SELECTED_WINDOW_MIN, MAX_DISTANCE_KM, VARIANT, RATIO_CUTOFF, LI_CUTOFF, VALIDATION_DIR, tag)


In [ ]:
plot_lidar_curtain(air, sat, hsrl_path, VALIDATION_DIR, tag)


## LI sensitivity against confident lidar cloud tops
This tests water-dominant versus ice-dominant tops only. It does not require a valid normalized ratio, and does not validate pure liquid throughout a cloud. No automatic “best” LI cutoff is adopted. A single flight is exploratory; uncertainty should eventually be estimated over cloud/flight segments, with separate flight days for validation.

In [ ]:
li_scores = av.liquid_top_sweep(footprints, np.arange(-0.2,1.01,0.02))
if not li_scores.empty:
    display(li_scores.iloc[::10])
    li_scores.plot(x="threshold",y=["recall","false_positive_rate","balanced_accuracy"],ylim=(0,1))
    plt.title("Exploratory liquid-top test; missing classes give undefined metrics");plt.show()
    li_scores.to_csv(VALIDATION_DIR/f"{tag}_LI_scores.csv",index=False)


## Optional P-3 in situ support
FCDP R1 reports concentration (#/L), LWC (g/m³), and size distributions. Its header states a **3-second time correction is already applied**; do not apply it again. FCDP alone cannot establish absence of ice or provide a complete phase label. Use quality-controlled imaging-probe evidence and aircraft altitude to interpret LTMP.

The optional cell downloads one FCDP table and inspects navigation choices. Set the navigation filename and field names from its printed header, including units; no guessed mapping is used. Other ICARTT FFI-1001 cloud-probe files can use the same reader. Particle-image ZIPs are deliberately not downloaded.

In [ ]:
RUN_INSITU = False
if RUN_INSITU:
    probe_path = av.download_exact(air_catalog["probes"], "ARCSIX-FCDP_P3B_20240610_R1.ict", AIRBORNE_DIR)
    probe = av.read_icartt(probe_path)
    print(probe.attrs["header"])
    probe["time"] = pd.Timestamp(DAY)+pd.to_timedelta(probe["Time_Start"],unit="s")
    display(probe[["time","conc","lwc"]].head())
    print("Navigation choices:", [av.name(g) for g in air_catalog["navigation"] if av.name(g).endswith(".ict")])


In [ ]:
# Edit only after inspecting the chosen navigation header.
NAV_FILENAME = "ARCSIX-MetNav_P3B_20240610_R0.ict"
NAV_FIELDS = dict(seconds="EDIT", latitude="EDIT", longitude="EDIT", altitude_m="EDIT")
if RUN_INSITU and NAV_FILENAME:
    nav_path = av.download_exact(air_catalog["navigation"], NAV_FILENAME, AIRBORNE_DIR)
    nav_raw = av.read_icartt(nav_path)
    print(nav_raw.attrs["header"])
    if any(v=="EDIT" for v in NAV_FIELDS.values()):
        print("Set NAV_FIELDS from the header and convert altitude to metres before matching.")
    else:
        nav = nav_raw[[*NAV_FIELDS.values()]].rename(columns={v:k for k,v in NAV_FIELDS.items()})
        nav["time"] = pd.Timestamp(DAY)+pd.to_timedelta(nav.pop("seconds"),unit="s")
        # Never bridge navigation gaps larger than one second.
        insitu = pd.merge_asof(probe.dropna(subset=["time"]).sort_values("time"),
            nav.dropna(subset=["time"]).sort_values("time"),on="time",direction="nearest",tolerance=pd.Timedelta(seconds=1))
        insitu_matches = av.match_profiles(sat,insitu,SELECTED_WINDOW_MIN,MAX_DISTANCE_KM)
        insitu_matches.to_csv(VALIDATION_DIR/f"{tag}_FCDP_matches.csv",index=False)
        print("Matched in situ records:",len(insitu_matches))
        if not insitu_matches.empty:
            # One summary per satellite sample; retain vertical range rather than hide it.
            insitu_summary=insitu_matches.groupby("sat_sat_id").agg(
                lwc_median=("air_lwc","median"),altitude_min_m=("air_altitude_m","min"),
                altitude_max_m=("air_altitude_m","max"),n_records=("air_index","size"))
            display(insitu_summary)
            insitu_summary.to_csv(VALIDATION_DIR/f"{tag}_FCDP_summary.csv")


## Ratio-threshold tuning only after independent phase review
Create labels from lidar visibility, vertical structure and/or quality-controlled in situ ice/liquid evidence—not from LI or the OCI ratio itself. A liquid top alone is insufficient for a `liquid` (all-liquid) label. Preserve `unknown` for attenuated profiles or doubtful collocations. Ice far beneath the optically sensed cloud portion should be identified in the evidence notes, not assumed to affect the SWIR ratio.

The optional analysis below tests **LTMP versus liquid among independently water-topped, high-LI observations**. It does not optimize the whole three-class map. Assign complete cloud/flight segments to train or test; do not randomly divide adjacent profiles. Training selects a threshold; held-out data evaluate that fixed threshold once. One flight is still only a pilot, even with a segment split. No confidence interval is claimed here.

In [ ]:
labels_path = VALIDATION_DIR/f"{tag}_reviewed_labels.csv"
if not footprints.empty and not labels_path.exists():
    labels = footprints[["sat_id"]].copy()
    labels["reference_phase"]="unknown"  # liquid, LTMP, ice, unknown
    labels["evidence"]=""  # independent source, visibility, altitude, QA, collocation justification
    labels["split"]=""     # train or test; assign by whole cloud/flight segment
    labels["segment_id"]=""
    labels.to_csv(labels_path,index=False)
    print("Review template written:",labels_path)

RUN_REVIEWED_TUNING = False
if RUN_REVIEWED_TUNING and not footprints.empty:
    labels=pd.read_csv(labels_path)
    scores, reviewed=av.reviewed_ratio_sweep(footprints,labels,np.arange(.7,1.801,.01),LI_CUTOFF)
    if scores.empty:
        print("No independently reviewed eligible liquid/LTMP samples; threshold unchanged.")
    else:
        best=scores.loc[scores.balanced_accuracy.idxmax()]
        print("Exploratory TRAIN optimum:",best.to_dict())
        scores.plot(x="threshold",y=["recall","precision","false_positive_rate","balanced_accuracy"],ylim=(0,1));plt.show()
        held=reviewed[reviewed["split"]=="test"]
        if held.reference_phase.nunique()==2:
            print("HELD-OUT scores at frozen threshold:",av.binary_scores(held.reference_phase=="LTMP",held.ratio>=best.threshold))
        else:
            print("No two-class held-out test: this is fitting, not independent validation.")
        scores.to_csv(VALIDATION_DIR/f"{tag}_ratio_training_scores.csv",index=False)
